# 13 — APS: dual size distributions

**Theme:** an instrument that sizes the same particle two different ways at
once.

The TSI Aerodynamic Particle Sizer measures an **aerodynamic** diameter, from
how a particle accelerates in a flow, and an **optical** diameter, from how much
light it scatters. The two differ in a way that depends on particle shape and
refractive index — so comparing them says something about what the particles
*are*, not just how large they are.

An APS export therefore loads as `Aerosol3d`: two size distributions over time,
plus the correlation between them.

In [ ]:
import aerosoltools as at

aps = at.load_aps_file("../../tests/data/Sample_APS_correlated.txt")

print("type         :", type(aps).__name__)
print("is_correlated:", aps.is_correlated)

`is_correlated` distinguishes the two kinds of export. A correlated file
carries both axes and the joint distribution; an aerodynamic-only file loads as
a plain `Aerosol2D`, because there is no second axis to pair with.

In [ ]:
aero_only = at.load_aps_file("../../tests/data/Sample_APS_aero.txt")
print("aerodynamic-only export loads as:", type(aero_only).__name__)

## The two axes

Each axis is available as an ordinary size-resolved dataset, so everything from
the earlier notebooks works on it.

In [ ]:
print(f"aerodynamic: {len(aps.aerodynamic.bin_mids)} bins, "
      f"{aps.aerodynamic.bin_mids.min():.0f}-{aps.aerodynamic.bin_mids.max():.0f} nm")
print(f"optical    : {len(aps.optical.bin_mids)} bins, "
      f"{aps.optical.bin_mids.min():.0f}-{aps.optical.bin_mids.max():.0f} nm")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.5))
aps.aerodynamic.plot_psd(ax=ax, activities=["All data"])
aps.optical.plot_psd(ax=ax, activities=["All data"])
ax.legend(["aerodynamic", "optical"])
ax.set_title("The same particles, sized two ways")

Where the two curves sit relative to each other is the physical signal: for
spherical particles of known refractive index they coincide, and they separate
as particles become irregular or optically unusual.

## Working with one axis

`as_2d` returns an independent copy of one axis — modify it freely without
touching the parent.

In [ ]:
standalone = aps.as_2d("aerodynamic")
print(type(standalone).__name__, "with", len(standalone.bin_mids), "bins")

`axis_view` returns a *live* view instead, with activities kept in sync with
the parent. Use `axis_view` when you want marked activities to follow, and
`as_2d` when you want an independent object.

In [ ]:
view = aps.axis_view("optical")
print(type(view).__name__, "- activities stay synced with the parent")

## Comparing the two sizings

`plot_aero_vs_optical` shows the relationship between the two diameters
directly.

In [ ]:
fig, ax = aps.plot_aero_vs_optical()

## The correlation matrix

For each time step the instrument records a joint distribution: how many
particles fell in each (aerodynamic, optical) pair of bins. `correlation_cube`
returns that as a dense time × optical × aerodynamic array.

In [ ]:
cube = aps.correlation_cube()
print("type :", type(cube).__name__)
print("shape:", cube.data.shape if hasattr(cube, "data") else "see below")

In [ ]:
fig, ax = aps.plot_aero_optical_3d()

The joint distribution is what makes an APS more than two instruments in a box:
it says which optical size each aerodynamically-sized particle had, rather than
just giving two independent histograms.

## Summaries

Because each axis is a normal size-resolved dataset, mass fractions, statistics
and exposure metrics all work per axis.

In [ ]:
aps.aerodynamic.summarize_activities(metrics=["PNC"], stats=["mean", "median", "max"])

---

**Next:** [15 — ACSM](15-acsm.ipynb).